# Fraud Detection at Scale

Companion notebook for the [Fraud Detection lesson](https://ml-viz-ruby.vercel.app/courses/ml-in-practice/20-fraud-detection-at-scale).

**The idea in one sentence.** Fraud is a needle-in-a-haystack problem (often <0.1% of
transactions), so you evaluate with **AUPRC / recall-at-low-FPR** (not accuracy), engineer
**velocity features** (how fast is this account moving?), and use **graph propagation** to
spread suspicion across accounts that share devices/cards.

Three production techniques, from scratch:

- **Metrics for extreme imbalance:** accuracy is 99.9% for a do-nothing model; AUPRC and
  recall-at-fixed-FPR are what matter.
- **Velocity features:** transaction rate in a time window — a strong fraud signal.
- **Graph propagation:** guilt-by-association across shared-entity edges.

We build all three, **validate the accuracy trap and graph propagation**, then cover the
gotchas.

> **To save your work:** click the **Copy to Drive** button at the top, or go to File → Save a copy in Drive.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
plt.style.use('dark_background')
rng = np.random.default_rng(99)

## 1 — Class imbalance: accuracy vs AUPRC

At 0.1% fraud rate, a trivial "not fraud" classifier scores 99.9% accuracy. We compute the Precision-Recall curve to show it's useless.

In [ ]:
# Simulate 10,000 transactions, 0.1% fraud
n = 10_000
fraud_rate = 0.001
labels = (rng.random(n) < fraud_rate).astype(int)
print(f"Total frauds: {labels.sum()} / {n} ({100*fraud_rate}% fraud rate)")

# Trivial model: predict "not fraud" for all
trivial_scores = np.zeros(n)
trivial_preds = np.zeros(n, dtype=int)
trivial_acc = (trivial_preds == labels).mean()
print(f"Trivial model accuracy: {trivial_acc:.4f}")

# A slightly better model: predict fraud with some signal
# Score = velocity feature proxy (random but slightly correlated with fraud)
scores = rng.uniform(0, 1, n)
scores[labels == 1] += 0.4  # fraud transactions score slightly higher
scores = np.clip(scores, 0, 1)

# Precision-Recall curve
def precision_recall_curve(labels, scores):
    thresholds = np.linspace(0, 1, 200)
    precs, recs = [], []
    for t in thresholds:
        pred = (scores >= t).astype(int)
        tp = ((pred == 1) & (labels == 1)).sum()
        fp = ((pred == 1) & (labels == 0)).sum()
        fn = ((pred == 0) & (labels == 1)).sum()
        precs.append(tp / (tp + fp + 1e-9))
        recs.append(tp / (tp + fn + 1e-9))
    return np.array(precs), np.array(recs)

precs, recs = precision_recall_curve(labels, scores)
auprc = np.trapezoid(precs[::-1], recs[::-1])

plt.figure(figsize=(7,4))
plt.plot(recs, precs, color='#6366f1', lw=2)
plt.axhline(fraud_rate, ls='--', color='gray', label=f'Random baseline ({fraud_rate:.3f})')
plt.xlabel('Recall'); plt.ylabel('Precision')
plt.title(f'Precision-Recall Curve (AUPRC={auprc:.4f})')
plt.legend(); plt.tight_layout(); plt.show()
print(f"AUPRC: {auprc:.4f}  (random baseline ≈ {fraud_rate:.4f})")

### Validate: accuracy is useless at 0.1% fraud

A model that predicts "not fraud" for everything is 99.9% accurate and catches zero fraud.
We confirm the accuracy trap — the reason fraud teams report AUPRC and recall-at-FPR
instead.

In [ ]:
assert abs(trivial_acc - (1 - fraud_rate)) < 0.01, 'do-nothing accuracy ~ 1 - fraud rate'
caught = (trivial_preds & labels).sum()
print(f'trivial "never fraud" model: accuracy {trivial_acc:.4f}, fraud caught {caught}/{labels.sum()}')
assert caught == 0, 'the 99.9%-accurate model catches no fraud'
print('\n✅ at 0.1% fraud, accuracy is meaningless — evaluate AUPRC / recall at a fixed FPR')

## 2 — Velocity features

Velocity features count activity in rolling windows — the strongest signal for payment fraud.

In [ ]:
# Simulate transaction log: (user_id, amount, timestamp_minutes)
n_users = 50
n_transactions = 500
user_ids  = rng.integers(0, n_users, n_transactions)
amounts   = rng.exponential(50, n_transactions)
times_min = np.sort(rng.uniform(0, 1440, n_transactions))  # 24 hours in minutes

def compute_velocity(user_ids, amounts, times_min, window_min=60):
    """For each transaction, count # transactions by same user in last `window_min` minutes."""
    n = len(user_ids)
    velocity = np.zeros(n, dtype=int)
    for i in range(n):
        t_start = times_min[i] - window_min
        # find all transactions by same user in [t_start, times_min[i])
        mask = (
            (user_ids[:i] == user_ids[i]) &
            (times_min[:i] >= t_start)
        )
        velocity[i] = mask.sum()
    return velocity

velocity_60min = compute_velocity(user_ids, amounts, times_min, window_min=60)
print(f"Max transactions by one user in any 60-min window: {velocity_60min.max()}")
print(f"Mean velocity: {velocity_60min.mean():.2f}")
print(f"Transactions with velocity > 5: {(velocity_60min > 5).sum()}")

## 3 — Graph propagation (guilt-by-association)

A simple message-passing step: if a node is fraudulent, propagate its fraud score to neighbors in the entity-sharing graph.

In [ ]:
# Toy entity graph: 10 accounts, some share devices
# Adjacency: account_i shares device with account_j
edges = [(0,1), (1,2), (3,4), (5,6), (6,7), (7,8)]  # pairs sharing devices
n_accounts = 10

adj = np.zeros((n_accounts, n_accounts))
for i,j in edges:
    adj[i,j] = adj[j,i] = 1

# Initial fraud scores (from feature model)
fraud_scores = np.array([0.9, 0.1, 0.1, 0.8, 0.1, 0.1, 0.2, 0.1, 0.1, 0.05])
# Accounts 0 and 3 are likely fraudsters

def graph_propagate(scores, adj, alpha=0.3, n_steps=2):
    """Propagate fraud scores through adjacency matrix for n_steps steps."""
    s = scores.copy()
    for _ in range(n_steps):
        neighbor_avg = adj @ s / (adj.sum(1) + 1e-9)
        s = (1 - alpha) * s + alpha * neighbor_avg  # blend own score + neighbor avg
    return s

propagated = graph_propagate(fraud_scores, adj)
print("Account | Initial score | After graph propagation")
for i in range(n_accounts):
    print(f"  {i:2d}    |  {fraud_scores[i]:.3f}        | {propagated[i]:.3f}  {'← elevated by neighbor' if propagated[i] > fraud_scores[i]+0.05 else ''}")

### Validate: graph propagation spreads suspicion to connected accounts

Fraud rings share devices and cards. Propagating fraud scores along the shared-entity graph
raises the score of an account *connected to* a known-fraud account, even if its own
features looked clean — guilt by association. We confirm a clean account linked to a
high-risk one gets pulled up.

In [ ]:
# one step of propagation: each account's score += influence from fraud neighbours
propagated = fraud_scores + 0.5 * (adj @ fraud_scores) / np.maximum(adj.sum(1), 1)
print('account  own-score  after-propagation')
for i in [1, 2, 4]:                       # accounts adjacent to high-risk 0/3
    print(f'  {i}      {fraud_scores[i]:.2f}       {propagated[i]:.2f}')
# account 1 shares a device with high-risk account 0 -> its score should rise
assert propagated[1] > fraud_scores[1], 'an account linked to fraud should gain suspicion'
# an isolated low-risk account keeps its low score
assert np.isclose(propagated[9], fraud_scores[9]) or propagated[9] < propagated[1]
print('\n✅ graph propagation spreads suspicion across shared-entity edges (guilt by association)')

## Gotchas & tradeoffs

| Gotcha | Consequence |
|--------|-------------|
| **accuracy on imbalance** | 99.9% and worthless (verified); use AUPRC / recall-at-FPR |
| **label delay** | fraud labels arrive weeks late (chargebacks) — training data is stale |
| **adversarial drift** | fraudsters adapt; static models decay fast, retrain often |
| **false positives cost customers** | blocking legit users has real cost — tune the FPR budget |
| **graph propagation over-spreads** | too many hops implicate innocent accounts; limit propagation |

Demo: a tighter FPR budget catches less fraud — the operating-point trade-off.

In [ ]:
# The precision/recall knob for fraud: you operate at a fixed, low false-positive rate
# (blocking legit customers is costly), so you report RECALL AT that FPR. Lowering the
# allowed FPR catches less fraud — the operating-point trade-off. We sweep it.
def recall_at_fpr_demo(labels, scores, tfpr):
    order = np.argsort(-scores)
    fp = tp = 0; N = (labels == 0).sum(); P = (labels == 1).sum()
    for i in order:
        if labels[i] == 1: tp += 1
        else: fp += 1
        if fp / N >= tfpr:
            break
    return tp / max(P, 1)
for tfpr in [0.001, 0.01, 0.05]:
    print(f'FPR budget {tfpr:.1%}: recall = {recall_at_fpr_demo(labels, scores, tfpr):.2f}')
print('\nA tighter FPR budget (fewer legit customers blocked) catches less fraud -> pick the operating point.')

## ✏️ Your turn

**Exercise.** Implement `recall_at_fpr(labels, scores, target_fpr)`:
Given an array of true labels and model scores, find the recall achievable at the given False Positive Rate (FPR) target.

This is the standard operational metric: "what fraction of fraud do we catch at a 1% false alarm rate?

In [ ]:
def recall_at_fpr(labels, scores, target_fpr=0.01):
    """
    labels: 1D array, 1=fraud, 0=not fraud
    scores: 1D float array, higher = more likely fraud
    target_fpr: desired FPR threshold (e.g., 0.01 = 1%)
    Returns: recall at that FPR.
    """
    # TODO(you): sweep thresholds, find the one where FPR ≈ target_fpr,
    # and return the corresponding recall (TPR)
    return ...

r = recall_at_fpr(labels, scores, target_fpr=0.05)
print(f"Recall @ 5% FPR: {r:.4f}")

In [ ]:
# Assertion
def _ref(labels, scores, tfpr=0.05):
    thresholds = np.linspace(0,1,500)
    best_rec, best_diff = 0, 1
    for t in thresholds:
        pred = (scores >= t).astype(int)
        fp = ((pred==1)&(labels==0)).sum()
        tn = ((pred==0)&(labels==0)).sum()
        tp = ((pred==1)&(labels==1)).sum()
        fn = ((pred==0)&(labels==1)).sum()
        fpr = fp/(fp+tn+1e-9)
        if abs(fpr-tfpr) < best_diff:
            best_diff = abs(fpr-tfpr)
            best_rec = tp/(tp+fn+1e-9)
    return best_rec
ref = _ref(labels, scores)
res = recall_at_fpr(labels, scores)
assert abs(res - ref) < 0.1, f"Expected ~{ref:.3f}, got {res:.3f}"
print(f"✓ recall_at_fpr correct: {res:.4f}")

<details><summary>Solution</summary>

```python
def recall_at_fpr(labels, scores, target_fpr=0.01):
    thresholds = np.linspace(0, 1, 500)
    best_recall, best_diff = 0.0, 1.0
    for t in thresholds:
        pred = (scores >= t).astype(int)
        fp = ((pred == 1) & (labels == 0)).sum()
        tn = ((pred == 0) & (labels == 0)).sum()
        tp = ((pred == 1) & (labels == 1)).sum()
        fn = ((pred == 0) & (labels == 1)).sum()
        fpr = fp / (fp + tn + 1e-9)
        if abs(fpr - target_fpr) < best_diff:
            best_diff = abs(fpr - target_fpr)
            best_recall = tp / (tp + fn + 1e-9)
    return best_recall
```
</details>

## Key takeaways

- **Fraud is extreme imbalance:** accuracy is 99.9% and useless (verified) — report AUPRC
  and **recall at a fixed low FPR**.
- **Velocity features** (transactions per time window) are a strong signal — sudden bursts
  flag account takeover.
- **Graph propagation spreads suspicion** across shared-device/card edges (verified) —
  guilt by association catches rings individual features miss.
- **You operate at a chosen FPR:** blocking legit customers is costly, so tighter FPR
  budgets catch less fraud (demo) — a business trade-off.